# Dataset - DataFrame

### 1. Preparar el entorno e importar la clase de carga
El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar src/clases_po y se agrega src a sys.path. Así se puede importar HFDatasetLoader sin instalar el proyecto como paquete.

In [1]:
import sys
import pandas as pd
from pathlib import Path
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader, Reportes

/home/runner/work/Machine-Learning-/Machine-Learning-/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.1. Configurar el dataset y la salida 
dataset_name identifica el repositorio en Hugging Face y split="train" indica la división que se descargará. El parámetro filename hace que el DataFrame también se guarde como CSV en src/carga_datos/datos/ cuando el notebook se ejecuta desde esa carpeta.

In [2]:
loader = HFDatasetLoader(
    dataset_name="sayanroy058/Business-Sales",
    split="train",
    filename="datos/Business-Sales.csv",
    
)

### 1.2 Descargar el dataset y crear el DataFrame
El método load() descarga el dataset, lo convierte a pandas.DataFrame y lo asigna a la variable df. La expresión df.head() muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df_raw = loader.load()
df_raw.head()

Generating train split:   0%|          | 0/558837 [00:00<?, ? examples/s]

Generating train split:   7%|▋         | 40000/558837 [00:00<00:01, 277925.85 examples/s]

Generating train split:  14%|█▍        | 80000/558837 [00:00<00:01, 302095.07 examples/s]

Generating train split:  21%|██▏       | 120000/558837 [00:00<00:01, 306125.79 examples/s]

Generating train split:  29%|██▊       | 160000/558837 [00:00<00:01, 313131.09 examples/s]

Generating train split:  36%|███▌      | 200000/558837 [00:00<00:01, 316015.59 examples/s]

... (7 outputs left out)

Generating train split:  93%|█████████▎| 520000/558837 [00:01<00:00, 311922.51 examples/s]

Generating train split: 100%|██████████| 558837/558837 [00:01<00:00, 313839.32 examples/s]

Generating train split: 100%|██████████| 558837/558837 [00:01<00:00, 311439.91 examples/s]

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg566472,ca,5.0,16639.0,white,black,kia motors america inc,20500.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
1,2015,Kia,Sorento,LX,SUV,automatic,5xyktca69fg561319,ca,5.0,9393.0,white,beige,kia motors america inc,20800.0,21500.0,Tue Dec 16 2014 12:30:00 GMT-0800 (PST)
2,2014,BMW,3 Series,328i SULEV,Sedan,automatic,wba3c1c51ek116351,ca,45.0,1331.0,gray,black,financial services remarketing (lease),31900.0,30000.0,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
3,2015,Volvo,S60,T5,Sedan,automatic,yv1612tb4f1310987,ca,41.0,14282.0,white,black,volvo na rep/world omni,27500.0,27750.0,Thu Jan 29 2015 04:30:00 GMT-0800 (PST)
4,2014,BMW,6 Series Gran Coupe,650i,Sedan,automatic,wba6b2c57ed129731,ca,43.0,2641.0,gray,black,financial services remarketing (lease),66000.0,67000.0,Thu Dec 18 2014 12:30:00 GMT-0800 (PST)


### 1.3 Traducir los nombres de las columnas
Se renombran las columnas al español para facilitar su lectura. Se usan minúsculas, sin tildes ni `ñ` y con `_` entre palabras, para evitar problemas de codificación al exportar o leer los datos. `odometer` se llama `millas` porque esa es su unidad original; en el paso 2.3 se convierte a kilómetros. `mmr` (Manheim Market Report) es el precio estimado de mercado.

In [4]:

#Nota: vin = Núero de identifcación del vehiculo

df = df_raw.rename(columns={
    "year": "anio_modelo",
    "make": "marca",
    "model": "modelo",
    "trim": "version",
    "body": "carroceria",
    "transmission": "transmision",
    "vin": "vin",
    "state": "estado",
    "condition": "condicion",
    "odometer": "millas",
    "color": "color_exterior",
    "interior": "color_interior",
    "seller": "vendedor",
    "mmr": "precio_mercado",
    "sellingprice": "precio_venta",
    "saledate": "fecha_venta",
})

df.columns.tolist()

['anio_modelo',
 'marca',
 'modelo',
 'version',
 'carroceria',
 'transmision',
 'vin',
 'estado',
 'condicion',
 'millas',
 'color_exterior',
 'color_interior',
 'vendedor',
 'precio_mercado',
 'precio_venta',
 'fecha_venta']

### 2. Limpieza de datos
Antes de limpiar se revisa la estructura original: 558.837 filas y 16 columnas. Cada fila es la venta de un vehículo usado en subasta, con sus características (año, marca, modelo, carrocería, millas recorridas, estado), el precio de referencia del mercado (`precio_mercado`) y el precio final de venta (`precio_venta`).

### 2.1 Corregir tipos y formatos
- `fecha_venta` llega como texto (`Tue Dec 16 2014 12:30:00 GMT-0800 (PST)`). Se toma la parte de fecha y hora y se convierte a `datetime`. Las filas que no se pueden convertir son 12 con fecha vacía y 26 registros desplazados (cada valor quedó en la columna siguiente, p. ej. `transmision = "Sedan"`), por lo que se eliminan.
- `condicion` mezcla dos escalas: 1–5 y 10–49. Los valores ≤ 5 se multiplican por 10 para dejar toda la columna en la escala 10–50.

In [5]:
df["fecha_venta"] = pd.to_datetime(df["fecha_venta"].str[4:24], format="%b %d %Y %H:%M:%S", errors="coerce")
df = df.dropna(subset=["fecha_venta"])

df["condicion"] = df["condicion"].where(df["condicion"] > 5, df["condicion"] * 10)

df[["fecha_venta", "condicion"]].describe()

,fecha_venta,condicion
count,558799,547005.000000
mean,2015-03-06 07:51:38.414993920,34.245343
min,2014-01-01 09:15:00,10.000000
25%,2015-01-21 02:00:00,27.000000
50%,2015-02-13 03:20:00,36.000000
75%,2015-05-22 02:00:00,42.000000
max,2015-07-21 02:30:00,50.000000
std,NaN,9.494347


### 2.2 Valores faltantes, duplicados e inconsistencias
En `color_exterior` y `color_interior` el valor faltante viene codificado como `"—"`, así que primero se convierte a `NaN` para contarlo junto con los demás nulos.

In [6]:
colores = ["color_exterior", "color_interior"]
df[colores] = df[colores].replace("—", np.nan)

df.isna().sum()[lambda s: s > 0]

marca             10301
modelo            10399
version           10651
carroceria        13195
transmision       65351
condicion         11794
millas               94
color_exterior    25434
color_interior    17824
dtype: int64

Tratamiento:
- **Se eliminan** las filas sin `marca`, `modelo`, `carroceria`, `millas`, `precio_mercado` o `precio_venta`. Son variables clave para estimar el precio y los nulos suman menos del 3 % de las filas.
- **Se rellenan con `"unknown"` (desconocido)** los vacíos de `transmision`, `color_exterior` y `color_interior`. En `transmision` falta el dato en cerca del 12 % de las filas; borrarlas haría perder muchos registros. Tampoco se rellenan con el valor más común (por ejemplo, `automatic`), porque eso inventaría datos y haría parecer que hay más vehículos de ese tipo de los que realmente hay.
- **Se rellenan con la mediana** los vacíos de `condicion` (cerca del 2 % de las filas). Como es un número, no se puede usar `"unknown"`. La mediana es el valor del medio cuando se ordenan todas las calificaciones (36). Se prefiere al promedio (34,2) porque hay un grupo de vehículos en muy mal estado que jala el promedio hacia abajo; la mediana no se deja afectar por esos casos extremos y representa mejor a un vehículo típico.
- `version` no se trata porque se descarta en el paso 2.4.
- **Duplicados:** no hay filas idénticas. Un mismo `vin` puede repetirse cuando el vehículo se revendió en otra fecha, lo cual es válido; solo se eliminan los registros con el mismo `vin` y la misma `fecha_venta`.
- **Inconsistencias:** `millas = 999999` es un valor de relleno y los precios de venta menores a 100 USD no son ventas reales, por lo que se eliminan.

In [7]:
df = df.dropna(subset=["marca", "modelo", "carroceria", "millas", "precio_mercado", "precio_venta"])
df[["transmision", *colores]] = df[["transmision", *colores]].fillna("unknown")
df["condicion"] = df["condicion"].fillna(df["condicion"].median())

df = df.drop_duplicates(subset=["vin", "fecha_venta"])
df = df[(df["millas"] < 999_999) & (df["precio_venta"] >= 100)]

df.shape

(545274, 16)

### 2.3 Normalizar categorías y unidades
- Las columnas categóricas combinan mayúsculas y minúsculas (`Sedan` / `sedan`, `SUV` / `suv`), así que se pasan a minúsculas y se quitan espacios.
- En `marca` hay varios nombres para una misma marca (`vw`, `ford tk`, `mercedes-b`, etc.); se unifican.
- `millas` se convierte a kilómetros (1 mi = 1,60934 km), la unidad usada en Colombia, y la columna pasa a llamarse `kilometraje`.

In [8]:
categoricas = ["marca", "modelo", "carroceria", "transmision", "estado", *colores]
df[categoricas] = df[categoricas].apply(lambda s: s.str.strip().str.lower())

df["marca"] = df["marca"].replace({
    "chev truck": "chevrolet",
    "dodge tk": "dodge",
    "ford tk": "ford",
    "ford truck": "ford",
    "gmc truck": "gmc",
    "hyundai tk": "hyundai",
    "mazda tk": "mazda",
    "landrover": "land rover",
    "mercedes": "mercedes-benz",
    "mercedes-b": "mercedes-benz",
    "vw": "volkswagen",
})

df["millas"] = (df["millas"] * 1.60934).round()
df = df.rename(columns={"millas": "kilometraje"})

df[categoricas].nunique()

marca              53
modelo            772
carroceria         45
transmision         3
estado             38
color_exterior     20
color_interior     17
dtype: int64

### 2.4 Seleccionar columnas y registros
El alcance del proyecto es estimar el precio de venta y analizar la depreciación. Se descartan:
- `vin`: identificador único del vehículo; ya se usó para revisar duplicados y no aporta al análisis.
- `version`: casi 2.000 versiones distintas y con nulos; `modelo` ya describe el vehículo.
- `vendedor`: texto libre con más de 14.000 valores distintos.

Los registros que quedan son los válidos después de los pasos anteriores.

In [9]:
df = df.drop(columns=["vin", "version", "vendedor"]).reset_index(drop=True)

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545274 entries, 0 to 545273
Data columns (total 13 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   anio_modelo     545274 non-null  int64         
 1   marca           545274 non-null  object        
 2   modelo          545274 non-null  object        
 3   carroceria      545274 non-null  object        
 4   transmision     545274 non-null  object        
 5   estado          545274 non-null  object        
 6   condicion       545274 non-null  float64       
 7   kilometraje     545274 non-null  float64       
 8   color_exterior  545274 non-null  object        
 9   color_interior  545274 non-null  object        
 10  precio_mercado  545274 non-null  float64       
 11  precio_venta    545274 non-null  float64       
 12  fecha_venta     545274 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(4), int64(1), object(7)
memory usage: 54.1+ MB


,anio_modelo,marca,modelo,carroceria,transmision,estado,condicion,kilometraje,color_exterior,color_interior,precio_mercado,precio_venta,fecha_venta
0,2015,kia,sorento,suv,automatic,ca,50.0,26778.0,white,black,20500.0,21500.0,2014-12-16 12:30:00
1,2015,kia,sorento,suv,automatic,ca,50.0,15117.0,white,beige,20800.0,21500.0,2014-12-16 12:30:00
2,2014,bmw,3 series,sedan,automatic,ca,45.0,2142.0,gray,black,31900.0,30000.0,2015-01-15 04:30:00
3,2015,volvo,s60,sedan,automatic,ca,41.0,22985.0,white,black,27500.0,27750.0,2015-01-29 04:30:00
4,2014,bmw,6 series gran coupe,sedan,automatic,ca,43.0,4250.0,gray,black,66000.0,67000.0,2014-12-18 12:30:00


### 3. Enriquecimiento de la información
Las nuevas variables se calculan con las columnas que ya tiene el dataset, así que no se une ninguna tabla externa y el número de registros no cambia. Cada variable apoya una o varias de las preguntas de negocio de la sección 4 (P1 a P5).

### 3.1 Periodo y antigüedad
- `mes_venta`: mes en que se vendió el vehículo. Las ventas van de diciembre de 2014 a julio de 2015 (más 201 registros de enero de 2014).
- `antiguedad`: años que tenía el vehículo al venderse. Un modelo 2015 vendido en diciembre de 2014 daría −1, porque los modelos salen al mercado antes de su año; en ese caso se deja en 0.

In [10]:
df["mes_venta"] = df["fecha_venta"].dt.to_period("M")
df["antiguedad"] = (df["fecha_venta"].dt.year - df["anio_modelo"]).clip(lower=0)

df["antiguedad"].describe()

count    545274.000000
mean          4.762021
std           3.863390
min           0.000000
25%           2.000000
50%           3.000000
75%           7.000000
max          25.000000
Name: antiguedad, dtype: float64

### 3.2 Comparación con el precio de mercado
`precio_mercado` (MMR) es lo que se espera que valga el vehículo. Compararlo con `precio_venta` muestra si se vendió caro o barato.
- `pct_vs_mercado`: porcentaje que se pagó por encima (+) o por debajo (−) del precio de mercado. Al estar en porcentaje permite comparar carros baratos y caros en la misma escala.
- `resultado_venta`: clasifica la venta. Una diferencia de ±5 % se considera normal porque el MMR es una estimación, no un precio exacto.

In [11]:
df["pct_vs_mercado"] = ((df["precio_venta"] - df["precio_mercado"]) / df["precio_mercado"] * 100).round(1)
df["resultado_venta"] = pd.cut(
    df["pct_vs_mercado"],
    bins=[-np.inf, -5, 5, np.inf],
    labels=["por debajo", "en el mercado", "por encima"],
)

df["resultado_venta"].value_counts(normalize=True).round(3)

resultado_venta
en el mercado    0.425
por debajo       0.311
por encima       0.264
Name: proportion, dtype: float64

### 3.3 Rango de condición
- `rango_condicion`: convierte la calificación numérica de `condicion` (10–50) en cuatro niveles fáciles de leer.

In [12]:
df["rango_condicion"] = pd.cut(
    df["condicion"],
    bins=[0, 19, 29, 39, 50],
    labels=["mala", "regular", "buena", "excelente"],
)

df["rango_condicion"].value_counts()

rango_condicion
excelente    188801
buena        180265
regular      127721
mala          48487
Name: count, dtype: int64

### 3.4 Precio relativo
- `precio_relativo`: compara el precio de venta de cada vehículo con el de otros iguales (misma marca, modelo y año). Se calcula dividiendo `precio_venta` por la mediana de `precio_venta` de ese grupo. Un valor de 1,10 significa que se vendió 10 % más caro que un vehículo típico igual a él; 0,90, 10 % más barato.
- Al comparar solo vehículos iguales, las diferencias de precio que quedan se pueden atribuir al kilometraje y a la condición, sin que influyan la marca, el modelo ni la antigüedad.
- Solo usa precios reales de venta del dataset, no el MMR.
- Si el grupo tiene menos de 5 vehículos, se deja vacío: con tan pocos casos la mediana no es una referencia confiable (en un grupo de 1 vehículo el valor siempre sería 1). Esto afecta a 1.865 filas (0,3 %).

In [13]:
grupo = df.groupby(["marca", "modelo", "anio_modelo"])["precio_venta"]
df["precio_relativo"] = (df["precio_venta"] / grupo.transform("median")).round(3)
df["precio_relativo"] = df["precio_relativo"].where(grupo.transform("size") >= 5)

df["precio_relativo"].describe()

count    543409.000000
mean          1.021692
std           0.316527
min           0.023000
25%           0.878000
50%           1.000000
75%           1.131000
max          24.727000
Name: precio_relativo, dtype: float64

### 3.5 Resumen de las variables incorporadas

| Variable | Fórmula o regla | Significado | Pregunta |
|---|---|---|---|
| `mes_venta` | Mes de `fecha_venta` (`AAAA-MM`) | Periodo en que se vendió | P3 |
| `antiguedad` | Año de `fecha_venta` − `anio_modelo`, mínimo 0 | Años de uso al venderse | P1 |
| `pct_vs_mercado` | (`precio_venta` − `precio_mercado`) / `precio_mercado` × 100 | Porcentaje por encima (+) o por debajo (−) del mercado | P2, P4 (a través de `resultado_venta`) |
| `resultado_venta` | ≤ −5 %: por debajo · −5 % a 5 %: en el mercado · > 5 %: por encima | Si la venta fue cara, justa o barata | P2, P4 |
| `rango_condicion` | 10–19: mala · 20–29: regular · 30–39: buena · 40–50: excelente | Estado del vehículo en palabras | P2 |
| `precio_relativo` | `precio_venta` / mediana de `precio_venta` del grupo (misma `marca`, `modelo` y `anio_modelo`); vacío si el grupo tiene menos de 5 vehículos | Qué tan caro o barato se vendió frente a vehículos iguales | P5 |

In [14]:
nuevas = ["mes_venta", "antiguedad", "pct_vs_mercado",
          "resultado_venta", "rango_condicion", "precio_relativo"]

df[nuevas].head()
df.tail(10)

,anio_modelo,marca,modelo,carroceria,transmision,estado,condicion,kilometraje,color_exterior,color_interior,precio_mercado,precio_venta,fecha_venta,mes_venta,antiguedad,pct_vs_mercado,resultado_venta,rango_condicion,precio_relativo
545264,2014,jeep,grand cherokee,suv,automatic,pa,42.0,40523.0,gray,black,26000.0,24500.0,2015-07-07 06:30:00,2015-07,1,-5.8,por debajo,excelente,0.795
545265,2012,dodge,grand caravan,minivan,automatic,ma,37.0,156164.0,silver,gray,8300.0,7800.0,2015-07-07 06:30:00,2015-07,3,-6.0,por debajo,buena,0.804
545266,2012,hyundai,elantra,sedan,unknown,pa,40.0,107375.0,gray,gray,10250.0,10400.0,2015-07-08 07:30:00,2015-07,3,1.5,en el mercado,excelente,0.959
545267,2012,nissan,sentra,sedan,unknown,tn,26.0,57708.0,white,gray,9950.0,10400.0,2015-07-08 17:15:00,2015-07,3,4.5,en el mercado,regular,1.061
545268,2011,bmw,5 series,sedan,automatic,fl,39.0,106865.0,white,brown,20300.0,22800.0,2015-07-07 06:15:00,2015-07,4,12.3,por encima,buena,0.884
545269,2015,kia,k900,sedan,unknown,in,45.0,29379.0,silver,black,35300.0,33000.0,2015-07-09 07:00:00,2015-07,0,-6.5,por debajo,excelente,0.846
545270,2012,ram,2500,crew cab,automatic,wa,50.0,87537.0,white,black,30200.0,30800.0,2015-07-08 09:30:00,2015-07,3,2.0,en el mercado,excelente,1.150
545271,2012,bmw,x5,suv,automatic,ca,48.0,81370.0,black,black,29800.0,34000.0,2015-07-08 09:30:00,2015-07,3,14.1,por encima,excelente,1.023
545272,2015,nissan,altima,sedan,automatic,ga,38.0,26808.0,white,black,15100.0,11100.0,2015-07-09 06:45:00,2015-07,0,-26.5,por debajo,buena,0.740
545273,2014,ford,f-150,supercrew,automatic,ca,34.0,24153.0,gray,gray,29600.0,26700.0,2015-05-28 05:30:00,2015-05,1,-9.8,por debajo,buena,1.123


### 4. Preguntas de negocio
**P1. Depreciación por antigüedad:** ¿cuál es el precio promedio de venta según los años de antigüedad del vehículo y qué porcentaje de su valor conserva con el paso de los años?

**P2. Condición y precio de mercado:** según la condición del vehículo, ¿qué porcentaje de las ventas se hace por debajo, en el mercado o por encima de su precio de mercado?

**P3. Ventas por mes:** ¿en qué meses se vendieron más vehículos y cuál fue su precio promedio de venta?

**P4. Marcas frente al mercado:** ¿qué marcas tienen el mayor y el menor porcentaje de ventas por encima de su precio de mercado?

**P5. Kilometraje vs. condición:** al comparar vehículos de la misma marca, modelo y año, ¿qué se relaciona más con el precio de venta: el kilometraje o la condición del vehículo?

### 5. Respuestas a las preguntas de negocio
Cada respuesta agrupa los datos con una de las variables creadas en la sección 3 y calcula conteos, promedios o porcentajes. Los precios están en dólares estadounidenses (USD).

### 5.1 P1. Depreciación por antigüedad
**Nivel de detalle:** años de antigüedad, de 0 a 15 (después de 15 años hay muy pocas ventas).

**Operación:** se agrupa por `antiguedad`, se cuentan las ventas y se calcula el precio promedio. El valor conservado es el precio promedio de cada año dividido entre el precio promedio de los vehículos de 0 años, por 100.

In [15]:
p1 = df[df["antiguedad"] <= 15].groupby("antiguedad")["precio_venta"].agg(["count", "mean"])
p1.columns = ["Ventas", "Precio promedio (USD)"]
p1["Valor conservado (%)"] = p1["Precio promedio (USD)"] / p1["Precio promedio (USD)"].iloc[0] * 100
p1 = p1.round(1).rename_axis("Antigüedad (años)")

p1

,Ventas,Precio promedio (USD),Valor conservado (%)
Antigüedad (años),,,
0,14298,23966.8,100.0
1,81987,19807.2,82.6
2,98827,17388.3,72.6
3,98772,16589.2,69.2
4,45754,15657.8,65.3
5,25427,12323.1,51.4
6,21629,10287.5,42.9
7,31020,9377.9,39.1
8,28815,7741.9,32.3


**Hallazgos:** un vehículo de 0 años se vende en promedio a 23.966,8 USD. Después de 1 año conserva el 82,6 % de ese valor; a los 5 años, cerca de la mitad (51,4 %); a los 10 años, el 20,0 %; y a los 15 años, solo el 7,2 %. Las caídas más fuertes están en los dos primeros años y entre los 4 y los 5 años.

**Utilidad:** permite estimar cuánto vale un vehículo según su edad y fijar precios de compra y reventa.


### 5.2 P2. Condición y precio de mercado
**Nivel de detalle:** rango de condición (mala, regular, buena, excelente).

**Operación:** tabla cruzada (`pd.crosstab`) entre `rango_condicion` y `resultado_venta`, en porcentaje por fila. Cada fila suma 100 %.

In [16]:
p2 = pd.crosstab(df["rango_condicion"], df["resultado_venta"], normalize="index") * 100
p2 = p2.round(1).rename_axis(index="Condición", columns="Resultado de la venta (%)")

p2

Resultado de la venta (%),por debajo,en el mercado,por encima
Condición,,,
mala,74.5,10.1,15.4
regular,53.0,24.3,22.7
buena,25.0,46.6,28.4
excelente,11.0,59.3,29.7


**Hallazgos:** a mejor condición, menos ventas por debajo del mercado. En condición mala, el 74,5 % de los vehículos se vende por debajo del mercado; en condición excelente, solo el 11,0 %, y el 59,3 % se vende en el mercado.

**Utilidad:** un vendedor puede anticipar que un vehículo en mala condición casi siempre se venderá por debajo de su precio de mercado, y evaluar si le conviene repararlo antes de venderlo.



### 5.3 P3. Ventas por mes
**Nivel de detalle:** mes de venta.

**Operación:** se agrupa por `mes_venta`, se cuentan las ventas y se calcula el precio promedio. La participación es el porcentaje de ventas de cada mes sobre el total. Se excluyen los meses con menos de 1.000 ventas (enero y febrero de 2014), porque tienen muy pocos datos.

In [17]:
p3 = df.groupby("mes_venta")["precio_venta"].agg(["count", "mean"])
p3.columns = ["Ventas", "Precio promedio (USD)"]
p3 = p3[p3["Ventas"] >= 1000]
p3["Participación (%)"] = p3["Ventas"] / p3["Ventas"].sum() * 100
p3 = p3.round(1).rename_axis("Mes de venta")

p3

,Ventas,Precio promedio (USD),Participación (%)
Mes de venta,,,
2014-12,51718,11449.4,9.5
2015-01,137205,13405.6,25.2
2015-02,159047,13731.7,29.2
2015-03,45182,13564.6,8.3
2015-04,1402,10429.5,0.3
2015-05,51285,14432.8,9.4
2015-06,97971,15082.0,18.0
2015-07,1262,17051.5,0.2


**Hallazgos:** los meses con más ventas son febrero de 2015 (159.047 ventas, 29,2 %) y enero de 2015 (137.205 ventas, 25,2 %); juntos suman más de la mitad del total. El precio promedio sube de 11.449,4 USD en diciembre de 2014 a 15.082,0 USD en junio de 2015. Abril y julio de 2015 tienen muy pocas ventas (1.402 y 1.262), así que sus precios promedio no son confiables.

**Utilidad:** indica en qué meses hay más vehículos disponibles para comprar y en cuáles los precios son más altos.


### 5.4 P4. Marcas frente al mercado
**Nivel de detalle:** marca. Solo se incluyen marcas con al menos 1.000 ventas.

**Operación:** tabla cruzada entre `marca` y `resultado_venta`, en porcentaje por fila, ordenada de mayor a menor porcentaje de ventas por encima del mercado.

In [18]:
ventas_marca = df["marca"].value_counts()
marcas = ventas_marca[ventas_marca >= 1000].index

p4 = pd.crosstab(df["marca"], df["resultado_venta"], normalize="index") * 100
p4 = p4.loc[marcas].sort_values("por encima", ascending=False).round(1)
p4 = p4.rename_axis(index="Marca", columns="Resultado de la venta (%)")

p4

Resultado de la venta (%),por debajo,en el mercado,por encima
Marca,,,
suzuki,41.1,14.6,44.3
saturn,47.3,13.5,39.1
mercury,50.0,12.2,37.8
pontiac,50.8,13.6,35.6
mitsubishi,36.5,30.6,33.0
scion,34.4,33.0,32.5
honda,32.7,35.7,31.6
volkswagen,35.7,33.4,31.0
cadillac,34.6,34.6,30.8


**Hallazgos:** las marcas con más ventas por encima del mercado son Suzuki (44,3 %), Saturn (39,1 %), Mercury (37,8 %) y Pontiac (35,6 %). Pero también son de las que más se venden por debajo (entre 41 % y 51 %), es decir, sus precios son muy variables. Las marcas con menos ventas por encima son Ram (19,3 %), Land Rover (20,6 %), Mercedes-Benz (21,6 %) y Kia (21,7 %); Ram y Land Rover son además las que más se venden en el mercado (58,1 % y 55,6 %), así que su precio es el más predecible.

**Utilidad:** distingue las marcas con precio predecible de las marcas con precio variable, donde hay más riesgo y más oportunidad de comprar barato o vender caro.



### 5.5 P5. Kilometraje vs. condición
**Nivel de detalle:** factor (kilometraje y condición).

**Operación:** correlación entre `precio_relativo` y cada factor. Como `precio_relativo` ya compara cada vehículo con otros de la misma marca, modelo y año, el resultado no depende de la marca ni de la antigüedad. La correlación va de −1 a 1: el signo indica la dirección de la relación y, cuanto más lejos de 0, más fuerte es. Los 1.865 vehículos sin `precio_relativo` no se tienen en cuenta.

In [19]:
p5 = df[["kilometraje", "condicion"]].corrwith(df["precio_relativo"]).round(3).to_frame("Correlación con precio_relativo")
p5.index = ["Kilometraje", "Condición"]
p5 = p5.rename_axis("Factor")

p5

,Correlación con precio_relativo
Factor,
Kilometraje,-0.217
Condición,0.250


**Hallazgos:** el kilometraje tiene correlación negativa (−0,217): a más kilómetros, menor precio. La condición tiene correlación positiva (0,250): a mejor condición, mayor precio. Como 0,250 está un poco más lejos de 0 que −0,217, la condición se relaciona un poco más con el precio. Pero la diferencia es pequeña, y ambas relaciones son débiles.

**Utilidad:** el kilometraje no se puede cambiar, pero la condición sí. Reparar y presentar bien un vehículo antes de venderlo puede ayudar a recuperar parte de su valor.


### 6. Exportar el dataset preparado y las tablas de resultados
Cada ejecución sobrescribe los archivos, así que siempre quedan actualizados. Se guardan con codificación `utf-8-sig` para que Excel muestre bien las tildes.
- Dataset preparado: `src/carga_datos/datos/Business-Sales-preparado.csv`
- Tablas de resultados: `src/carga_datos/datos/resultados/`

In [20]:
CARPETA = ROOT / "src" / "carga_datos"
RESULTADOS = CARPETA / "datos" / "resultados"
RESULTADOS.mkdir(parents=True, exist_ok=True)

df.to_csv(CARPETA / "datos" / "Business-Sales-preparado.csv", index=False, encoding="utf-8-sig")

tablas = {
    "p1_depreciacion_por_antiguedad": p1,
    "p2_condicion_vs_mercado": p2,
    "p3_ventas_por_mes": p3,
    "p4_marcas_vs_mercado": p4,
    "p5_kilometraje_vs_condicion": p5,
}
for nombre, tabla in tablas.items():
    tabla.to_csv(RESULTADOS / f"{nombre}.csv", encoding="utf-8-sig")

sorted(archivo.name for archivo in RESULTADOS.glob("*.csv"))

['p1_depreciacion_por_antiguedad.csv',
 'p2_condicion_vs_mercado.csv',
 'p3_ventas_por_mes.csv',
 'p4_marcas_vs_mercado.csv',
 'p5_kilometraje_vs_condicion.csv']

### 7. Reporte descriptivo del DataFrame final
Se usa la clase `Reportes` del proyecto (`src/clases_po/reportes.py`) para generar `src/carga_datos/reportes/Business-Sales-preparado.txt`. El reporte incluye la cantidad de filas y columnas, los duplicados, el nombre y tipo de cada columna, los valores nulos y las estadísticas descriptivas de las columnas numéricas. Se regenera en cada ejecución.

In [21]:
ruta_reporte = Reportes().guardar_reporte(df, CARPETA / "reportes" / "Business-Sales-preparado.txt")

print(ruta_reporte.read_text(encoding="utf-8"))

REPORTE DEL DATAFRAME
Filas: 545274
Columnas: 19
Duplicados: 0

Columnas y tipos:
- anio_modelo: int64
- marca: object
- modelo: object
- carroceria: object
- transmision: object
- estado: object
- condicion: float64
- kilometraje: float64
- color_exterior: object
- color_interior: object
- precio_mercado: float64
- precio_venta: float64
- fecha_venta: datetime64[ns]
- mes_venta: period[M]
- antiguedad: int64
- pct_vs_mercado: float64
- resultado_venta: category
- rango_condicion: category
- precio_relativo: float64

Valores nulos por columna:
- anio_modelo: 0
- marca: 0
- modelo: 0
- carroceria: 0
- transmision: 0
- estado: 0
- condicion: 0
- kilometraje: 0
- color_exterior: 0
- color_interior: 0
- precio_mercado: 0
- precio_venta: 0
- fecha_venta: 0
- mes_venta: 0
- antiguedad: 0
- pct_vs_mercado: 0
- resultado_venta: 0
- rango_condicion: 0
- precio_relativo: 1865

Estadísticas numéricas:
       anio_modelo   condicion  kilometraje  precio_mercado  precio_venta  antiguedad  pct_vs_me